# 04 Signal Generation

This notebook creates the supervised forward-return target, trains the primary XGBoost classifier and benchmark models, and saves diagnostic plots for feature importance, SHAP, confusion matrix, and ROC behavior.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
import shap
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.signal_model import DirectionalSignalModel, create_targets
from src.utils import load_yaml_config

config = load_yaml_config(PROJECT_ROOT / "config" / "parameters.yaml")
processed_dir = PROJECT_ROOT / config["data"]["processed_data_dir"]
figures_dir = PROJECT_ROOT / config["reporting"]["figures_dir"]
figures_dir.mkdir(parents=True, exist_ok=True)

dataset = pd.read_csv(processed_dir / "nsei_regimes.csv", index_col=0, parse_dates=True)
dataset = create_targets(
    dataset,
    horizon=config["signal_model"]["target_horizon"],
    return_threshold=config["signal_model"]["return_threshold"],
).dropna(subset=["target_binary"])
feature_columns = [column for column in config["signal_model"]["feature_columns"] if column in dataset.columns]

split_index = int(len(dataset) * 0.8)
train = dataset.iloc[:split_index].copy()
test = dataset.iloc[split_index:].copy()


In [ ]:
model_names = ["xgboost", "random_forest", "logistic_regression"]
scores = {}

for model_name in model_names:
    model = DirectionalSignalModel(model_name=model_name, task_type="binary")
    model.fit(train, feature_columns=feature_columns, target_column="target_binary")
    prediction_frame = model.make_prediction_frame(test)
    scores[model_name] = model.evaluate(test["target_binary"], prediction_frame)

scores


In [ ]:
primary_model = DirectionalSignalModel(model_name="xgboost", task_type="binary")
primary_model.fit(train, feature_columns=feature_columns, target_column="target_binary")
predictions = primary_model.make_prediction_frame(test)
signal_output = test.join(predictions)
signal_output.to_csv(processed_dir / "nsei_signals.csv")

importance = primary_model.get_feature_importance()
importance.head(15).sort_values("importance").plot.barh(x="feature", y="importance", figsize=(10, 7), title="Feature Importance")
plt.tight_layout()
plt.savefig(figures_dir / "feature_importance.png", dpi=150)
plt.show()

ConfusionMatrixDisplay.from_predictions(test["target_binary"].astype(int), predictions["prediction"].astype(int))
plt.tight_layout()
plt.savefig(figures_dir / "confusion_matrix.png", dpi=150)
plt.show()

RocCurveDisplay.from_predictions(test["target_binary"].astype(int), predictions["up_probability"])
plt.tight_layout()
plt.savefig(figures_dir / "roc_curve.png", dpi=150)
plt.show()

shap_values, shap_sample = primary_model.compute_shap_values(test, max_samples=250)
shap.summary_plot(shap_values, shap_sample, show=False)
plt.tight_layout()
plt.savefig(figures_dir / "shap_summary.png", dpi=150)
plt.show()
